# nanochat base train + eval notebook

Notebook-first workflow for `scripts/base_train.py` **and** `scripts/base_eval.py` without manual CLI typing.

In [1]:
import torch._dynamo
torch._dynamo.disable()


## 1) Configure training
Set token budget + evaluation cadence. CORE eval is enabled by default via `core_metric_every`.


In [2]:
from math import ceil

TRAINING_CONFIG = {
    # Logging/runtime
    'run': 'nb-base-train',
    'data_dir': None,
    'checkpoints_dir': None,
    'device_type': '',  # ''=autodetect, or set 'cuda'/'cpu'/'mps'

    # Model
    'depth': 4, #20,
    'aspect_ratio': 64,
    'head_dim': 16, #128,
    'max_seq_len': 32,#2048,
    'window_pattern': 'SSSL',

    # Batch + training horizon
    'device_batch_size': 8,
    'total_batch_size': 524288//64,  # in tokens
    'training_tokens': 20 * (524288//64),  # <-- edit this to control total training budget

    # Optimization
    'embedding_lr': 0.3,
    'unembedding_lr': 0.004,
    'weight_decay': 0.2,
    'matrix_lr': 0.02,
    'scalar_lr': 0.5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.95,
    'warmup_ratio': 0.0,
    'warmdown_ratio': 0.5,
    'final_lr_frac': 0.0,

    # Evaluation/checkpoint cadence
    'eval_every': 20,#250,
    'eval_tokens': 2 * 524288,
    'core_metric_every': -1,
    'core_metric_max_per_task': 500,
    'sample_every': 500,
    'save_every': 500,

    # Extras
    'resume_from_step': -1,
    'model_tag': 'nb_d20',
    'fp8': False,
    'fp8_recipe': 'tensorwise',
}


TRAINING_CONFIG['num_iterations'] = ceil(TRAINING_CONFIG['training_tokens'] / TRAINING_CONFIG['total_batch_size'])
print('Planned iterations:', TRAINING_CONFIG['num_iterations'])
print('Planned tokens   :', TRAINING_CONFIG['num_iterations'] * TRAINING_CONFIG['total_batch_size'])
print('CORE in training?:', TRAINING_CONFIG['core_metric_every'])

Planned iterations: 20
Planned tokens   : 163840
CORE in training?: -1


## 2) Convert config to argv for base_train


In [3]:
def base_train_argv(cfg: dict) -> list[str]:
    flag_map = {
        'run': '--run', 'device_type': '--device-type',
        'data_dir': '--data-dir', 'checkpoints_dir': '--checkpoints-dir',
        'depth': '--depth', 'aspect_ratio': '--aspect-ratio', 'head_dim': '--head-dim',
        'max_seq_len': '--max-seq-len', 'window_pattern': '--window-pattern',
        'num_iterations': '--num-iterations',
        'device_batch_size': '--device-batch-size', 'total_batch_size': '--total-batch-size',
        'embedding_lr': '--embedding-lr', 'unembedding_lr': '--unembedding-lr',
        'weight_decay': '--weight-decay', 'matrix_lr': '--matrix-lr', 'scalar_lr': '--scalar-lr',
        'adam_beta1': '--adam-beta1', 'adam_beta2': '--adam-beta2',
        'warmup_ratio': '--warmup-ratio', 'warmdown_ratio': '--warmdown-ratio', 'final_lr_frac': '--final-lr-frac',
        'resume_from_step': '--resume-from-step',
        'eval_every': '--eval-every', 'eval_tokens': '--eval-tokens',
        'core_metric_every': '--core-metric-every', 'core_metric_max_per_task': '--core-metric-max-per-task',
        'sample_every': '--sample-every', 'save_every': '--save-every',
        'model_tag': '--model-tag', 'fp8_recipe': '--fp8-recipe',
    }
    argv = ['scripts.base_train']
    if cfg.get('fp8', False):
        argv.append('--fp8')
    for k, flag in flag_map.items():
        if k in cfg:
            argv.extend([flag, str(cfg[k])])
    return argv

train_argv = base_train_argv(TRAINING_CONFIG)
print(' '.join(train_argv[:14]), '...')

scripts.base_train --run nb-base-train --device-type  --depth 4 --aspect-ratio 64 --head-dim 16 --max-seq-len 32 --window-pattern ...


## 3) Run base_train (includes bpb eval + CORE eval per cadence)
`core_metric_every` and `eval_every` control in-training evaluation.


In [4]:
import runpy, sys
old_argv = sys.argv[:]
sys.argv = train_argv
try:
    runpy.run_module('scripts.base_train', run_name='__main__')
finally:
    sys.argv = old_argv

2026-03-08 23:00:26,674 - nanochat.common - INFO - Distributed world size: 1
2026-03-08 23:00:26,675 - nanochat.common - WARNING - Peak flops undefined for: NVIDIA GeForce RTX 3050 Ti Laptop GPU, MFU will show as 0%



                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Peak FLOPS (BF16): inf
COMPUTE_DTYPE: torch.bfloat16 (auto-detected: CUDA SM 86 (bf16 supported))


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/seqaeon/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Vocab size: 32,768
Model config:
{
  "sequence_len": 32,
  "vocab_size": 32768,
  "n_layer": 4,
  "n_head": 16,
  "n_kv_head": 16,
  "n_embd": 256,
  "use_moe": false,
  "use_perm": false,
  "moe_num_experts": 8,
  "moe_router_dim": 64,
  "moe_embed_dim": 64,
  "use_remix_linear": false,
  "remix_context_dim": 64,
  "remix_basis_size": 64,
  "remixed_linear_kwargs": {
    "use_basis_gate": true,
    "use_output_gate": true,
    "use_context": true
  },
  "num_experts": 8,
  "total_embed_dim": 64,
  "router_dim": 64,
  "capacity_factor": 1.0,
  "use_sparse_top_k": false,
  "top_k": 1,
  "routing_mode": "token_choice",
  "context_window": -1,
  "causal": true,
  "use_expert_mlp": true,
  "use_output_projection": true,
  "use_expert_bias": false,
  "dropout": 0.0,
  "use_shared_base": false,
  "shared_base_dim": 128,
  "use_vocab

/home/seqaeon/Downloads/nanochat/nanochat/tokenizer.py:405: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_bytes = torch.load(f, map_location=device)


Parameter counts:
wte                     : 8,388,608
value_embeds            : 16,777,216
lm_head                 : 8,388,608
transformer_matrices    : 3,146,752
research                : 0
scalars                 : 8
total                   : 36,701,192
Estimated FLOPs per token: 6.945792e+07
Scaling LRs by 0.1250 for batch size 8,192 (reference: 524,288)
Scaling weight decay from 0.200000 to 0.238635 for depth 4
Scaling the LR for the AdamW parameters ∝1/√(256/768) = 1.732051
Using user-provided number of iterations: 20
Total number of training tokens: 163,840
Tokens : Scaling params ratio: 0.01
Total training FLOPs estimate: 1.137999e+13
Tokens / micro-batch / rank: 8 x 32 = 256
Tokens / micro-batch: 256
Total batch size 8,192 => gradient accumulation steps: 32
Step 00000 | Validation bpb: 3.319357
step 00000/00020 (0.00%) | loss: 10.397788 | lrm: 1.00 | dt: 11421.33ms | tok/sec: 717 | bf16_mfu: 0.00 | epoch: 1 pq: 0 rg: 1 | total time: 0.00m
step 00001/00020 (5.00%) | loss: 10.386

2026-03-08 23:02:47,150 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/model_000020.pt
2026-03-08 23:02:47,151 - nanochat.checkpoint_manager - INFO - Saved metadata to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/meta_000020.json
2026-03-08 23:02:47,514 - nanochat.checkpoint_manager - INFO - Saved optimizer state to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/optim_000020_rank0.pt


Peak memory usage: 513.10MiB
Total training time: 0.22m
Minimum validation bpb: 2.679026


step,▁▁█
total_training_flops,▁▁█
total_training_time,▁▁█
train/dt,▁
train/loss,▁
train/lrm,▁
train/mfu,▁
train/tok_per_sec,▁
val/bpb,█▁
step,20
total_training_flops,11379985612800


## 4) Optional: run standalone base_eval.py after training
Use this when you want a full explicit eval pass after training completes.


In [5]:
# BASE_EVAL_CONFIG = {
#     'device_type': TRAINING_CONFIG['device_type'],
#     'model_tag': TRAINING_CONFIG['model_tag'],
#     'step': None,  # None = latest checkpoint
#     'eval': 'core,bpb,sample',
#     'max_per_task': 500,
#     'device_batch_size': 16,
#     'split_tokens': 8 * 524288,
# }

# def base_eval_argv(cfg: dict) -> list[str]:
#     flag_map = {
#         'device_type': '--device-type',
#         'model_tag': '--model-tag',
#         'step': '--step',
#         'eval': '--eval',
#         'max_per_task': '--max-per-task',
#         'device_batch_size': '--device-batch-size',
#         'split_tokens': '--split-tokens',
#     }
#     argv = ['scripts.base_eval']
#     for k, flag in flag_map.items():
#         if k not in cfg or cfg[k] is None:
#             continue
#         argv.extend([flag, str(cfg[k])])
#     return argv

# eval_argv = base_eval_argv(BASE_EVAL_CONFIG)
# print(' '.join(eval_argv))

In [6]:
# # Uncomment to run explicit post-training eval:
# import runpy, sys
# old_argv = sys.argv[:]
# sys.argv = eval_argv
# try:
#     runpy.run_module('scripts.base_eval', run_name='__main__')
# finally:
#     sys.argv = old_argv

## Notes
- If you set `core_metric_every=-1`, training will skip CORE eval (this was the main reason it can appear missing).
- For quick smoke tests, lower depth/sequence/batch/token budget dramatically.
- Multi-GPU distributed runs are still best launched from terminal with `torchrun`.


## 5) Research branch config (MoE / Permutation / RemixedLinear)

Use this when you want notebook-driven experiments aligned with the `contextlinear.ipynb` branches.


In [7]:
RESEARCH_ALLOWED_KEYS = {
    "use_moe", "num_experts", "total_embed_dim", "router_dim", "capacity_factor",
    "use_sparse_top_k", "top_k", "routing_mode", "context_window",
    "causal", "use_expert_mlp", "use_output_projection",
    "use_expert_bias", "dropout", "use_shared_base", "shared_base_dim",
    "use_vocab_prior", "expert_residual", "allow_replacement",
    "use_embed_refine", "target_dim", "selection_mode", "use_perm",
    "use_remixed_linear", "context_dim", "linear_basis_size", "remixed_linear_kwargs",
    "router_context_window", "router_causal", "router_num_heads", "router_num_queries",
    "router_n_layers", "router_use_vocab_prior",
    "use_pos_embed", "moe_use_abs_pos_embed",
}

RESEARCH_MODEL_CONFIG = {
    "use_moe": True,
    "use_perm": False,
    "num_experts": 8,
    "router_dim": 64,
    "target_dim": 64,
    "selection_mode": "soft",
    "allow_replacement": True,
    "use_remixed_linear": False,
    "context_dim": 64,
    "linear_basis_size": 64,
    "router_context_window": -1,
    "router_causal": True,
    "router_num_heads": 4,
    "router_num_queries": 8,
    "router_n_layers": 2,
    "router_use_vocab_prior": False,
    "use_pos_embed": False,
    "moe_use_abs_pos_embed": True,
    "remixed_linear_kwargs": {
        "use_basis_gate": True,
        "use_output_gate": True,
        "use_context": True,
    },
}

def research_to_gpt_kwargs(cfg: dict) -> dict:
    unexpected = sorted(set(cfg) - RESEARCH_ALLOWED_KEYS)
    if unexpected:
        print(f"Warning: unexpected research keys: {unexpected}")
    return {
        "use_moe": cfg.get("use_moe", False),
        "use_perm": cfg.get("use_perm", False),
        "num_experts": cfg.get("num_experts", 8),
        "router_dim": cfg.get("router_dim", 64),
        "target_dim": cfg.get("target_dim", 64),
        "selection_mode": cfg.get("selection_mode", "soft"),
        "allow_replacement": cfg.get("allow_replacement", True),
        "use_remixed_linear": cfg.get("use_remixed_linear", False),
        "context_dim": cfg.get("context_dim", 64),
        "linear_basis_size": cfg.get("linear_basis_size", 64),
        "router_context_window": cfg.get("router_context_window", -1),
        "router_causal": cfg.get("router_causal", True),
        "router_num_heads": cfg.get("router_num_heads", 4),
        "router_num_queries": cfg.get("router_num_queries", 8),
        "router_n_layers": cfg.get("router_n_layers", 2),
        "router_use_vocab_prior": cfg.get("router_use_vocab_prior", False),
        "use_pos_embed": cfg.get("use_pos_embed", False),
        "moe_use_abs_pos_embed": cfg.get("moe_use_abs_pos_embed", True),
        "remixed_linear_kwargs": cfg.get("remixed_linear_kwargs", {
            "use_basis_gate": True,
            "use_output_gate": True,
            "use_context": True,
        }),
    }

RESEARCH_GPT_KWARGS = research_to_gpt_kwargs(RESEARCH_MODEL_CONFIG)
RESEARCH_GPT_KWARGS




{'use_moe': True,
 'use_perm': False,
 'num_experts': 8,
 'router_dim': 64,
 'target_dim': 64,
 'selection_mode': 'soft',
 'allow_replacement': True,
 'use_remixed_linear': False,
 'context_dim': 64,
 'linear_basis_size': 64,
 'remixed_linear_kwargs': {'use_basis_gate': True,
  'use_output_gate': True,
  'use_context': True}}

### 5.1) Build train argv from `RESEARCH_GPT_KWARGS` and run

After you compute `RESEARCH_GPT_KWARGS`, run this cell to inject those values into `base_train.py` CLI args.


In [8]:
MOE_SCALE = 5.0  # applied when any research mode is active


def _resolve_research_lrs(training_cfg: dict, research_kwargs: dict, moe_scale: float) -> dict:
    # Apply MOE scale to all LR groups whenever any research branch is enabled.
    research_active = bool(
        research_kwargs.get("use_moe", False)
        or research_kwargs.get("use_perm", False)
        or research_kwargs.get("use_remixed_linear", False)
    )
    scale = float(moe_scale) if research_active else 1.0

    resolved = {
        "embedding_lr": float(training_cfg["embedding_lr"]) * scale,
        "unembedding_lr": float(training_cfg["unembedding_lr"]) * scale,
        "matrix_lr": float(training_cfg["matrix_lr"]) * scale,
        "scalar_lr": float(training_cfg["scalar_lr"]) * scale,
    }

    # Research-specific LR overrides take precedence over both base LR and MOE_SCALE.
    # Note: `research_scaler_lr` is the canonical key for scalar LR overrides.
    override_map = {
        "embedding_lr": ["research_embedding_lr"],
        "unembedding_lr": ["research_unembedding_lr"],
        "matrix_lr": ["research_matrix_lr"],
        "scalar_lr": ["research_scaler_lr", "research_scalar_lr"],  # support legacy alias too
    }
    for target_key, candidates in override_map.items():
        override_val = next((research_kwargs[k] for k in candidates if research_kwargs.get(k, None) is not None), None)
        if override_val is not None:
            resolved[target_key] = float(override_val)
    return resolved


def _resolve_schedule(training_cfg: dict, research_kwargs: dict) -> tuple[float, float, float]:
    warmup = float(research_kwargs.get("research_warmup", training_cfg.get("warmup_ratio", 0.0)))
    one_cycle = bool(research_kwargs.get("research_one_cycle", False))
    if one_cycle:
        warmdown = max(0.0, 1.0 - warmup)
        final_lr_frac = 0.0
    else:
        warmdown = float(training_cfg.get("warmdown_ratio", 0.5))
        final_lr_frac = float(training_cfg.get("final_lr_frac", 0.0))
    return warmup, warmdown, final_lr_frac


def research_train_argv(training_cfg: dict, research_kwargs: dict, moe_scale: float = MOE_SCALE) -> list[str]:
    argv = base_train_argv(training_cfg)

    # bool flags
    if research_kwargs.get("use_moe", False):
        argv += ["--use-moe"]
    if research_kwargs.get("use_perm", False):
        argv += ["--use-perm"]
    if research_kwargs.get("allow_replacement", False):
        argv += ["--allow-replacement"]
    if research_kwargs.get("use_remixed_linear", False):
        argv += ["--use-remixed-linear"]

    # scalar args
    scalar_map = {
        "num_experts": "--num-experts",
        "router_dim": "--router-dim",
        "target_dim": "--target-dim",
        "selection_mode": "--selection-mode",
        "context_dim": "--context-dim",
        "linear_basis_size": "--linear-basis-size",
    }
    for k, flag in scalar_map.items():
        if k in research_kwargs and research_kwargs[k] is not None:
            argv += [flag, str(research_kwargs[k])]

    # remixed linear gate controls
    remix_kwargs = research_kwargs.get("remixed_linear_kwargs", {})
    argv += ["--remix-use-basis-gate", str(int(bool(remix_kwargs.get("use_basis_gate", True))))]
    argv += ["--remix-use-output-gate", str(int(bool(remix_kwargs.get("use_output_gate", True))))]
    argv += ["--remix-use-context", str(int(bool(remix_kwargs.get("use_context", True))))]

    resolved_lrs = _resolve_research_lrs(training_cfg, research_kwargs, moe_scale)
    argv += ["--embedding-lr", str(resolved_lrs["embedding_lr"])]
    argv += ["--unembedding-lr", str(resolved_lrs["unembedding_lr"])]
    argv += ["--matrix-lr", str(resolved_lrs["matrix_lr"])]
    argv += ["--scalar-lr", str(resolved_lrs["scalar_lr"])]

    # Research-specific beta overrides (if provided)
    beta1 = float(research_kwargs.get("research_adam_beta1", training_cfg.get("adam_beta1", 0.8)))
    beta2 = float(research_kwargs.get("research_adam_beta2", training_cfg.get("adam_beta2", 0.95)))
    argv += ["--adam-beta1", str(beta1), "--adam-beta2", str(beta2)]

    warmup, warmdown, final_lr_frac = _resolve_schedule(training_cfg, research_kwargs)
    argv += [
        "--warmup-ratio", str(warmup),
        "--warmdown-ratio", str(warmdown),
        "--final-lr-frac", str(final_lr_frac),
    ]

    if bool(research_kwargs.get("research_use_pos_embed", research_kwargs.get("use_pos_embed", False))):
        argv += ["--use-pos-embed"]
    argv += ["--moe-use-abs-pos-embed", str(int(bool(research_kwargs.get("moe_use_abs_pos_embed", True))))]

    return argv


# For research sweeps keep sequence length and target dim fixed for comparability.
TRAINING_CONFIG['max_seq_len'] = 64
RESEARCH_GPT_KWARGS['target_dim'] = 64
# Ensure attention divisibility with target_dim=64
TRAINING_CONFIG['head_dim'] = 16

research_argv = research_train_argv(TRAINING_CONFIG, RESEARCH_GPT_KWARGS, moe_scale=MOE_SCALE)
print(' '.join(research_argv))






scripts.base_train --run nb-base-train --device-type  --depth 4 --aspect-ratio 64 --head-dim 64 --max-seq-len 32 --window-pattern SSSL --num-iterations 20 --device-batch-size 8 --total-batch-size 8192 --embedding-lr 0.3 --unembedding-lr 0.004 --weight-decay 0.2 --matrix-lr 0.02 --scalar-lr 0.5 --adam-beta1 0.8 --adam-beta2 0.95 --warmup-ratio 0.0 --warmdown-ratio 0.5 --final-lr-frac 0.0 --resume-from-step -1 --eval-every 20 --eval-tokens 1048576 --core-metric-every -1 --core-metric-max-per-task 500 --sample-every 500 --save-every 500 --model-tag nb_d20 --fp8-recipe tensorwise --use-moe --allow-replacement --num-experts 8 --router-dim 64 --target-dim 64 --selection-mode soft --context-dim 64 --linear-basis-size 64 --remix-use-basis-gate 1 --remix-use-output-gate 1 --remix-use-context 1 --embedding-lr 1.5 --unembedding-lr 0.02 --matrix-lr 0.1 --scalar-lr 2.5


In [9]:
# Run research training from notebook:
import runpy, sys
old_argv = sys.argv[:]
sys.argv = research_argv
try:
    runpy.run_module('scripts.base_train', run_name='__main__')
finally:
    sys.argv = old_argv


2026-03-08 23:02:49,246 - nanochat.common - INFO - Distributed world size: 1
2026-03-08 23:02:49,247 - nanochat.common - WARNING - Peak flops undefined for: NVIDIA GeForce RTX 3050 Ti Laptop GPU, MFU will show as 0%



                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Peak FLOPS (BF16): inf
COMPUTE_DTYPE: torch.bfloat16 (auto-detected: CUDA SM 86 (bf16 supported))


wandb: Currently logged in as: seqaeon (seqaeon-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Vocab size: 32,768
Model config:
{
  "sequence_len": 32,
  "vocab_size": 32768,
  "n_layer": 4,
  "n_head": 1,
  "n_kv_head": 1,
  "n_embd": 64,
  "use_moe": true,
  "use_perm": false,
  "moe_num_experts": 8,
  "moe_router_dim": 64,
  "moe_embed_dim": 64,
  "use_remix_linear": false,
  "remix_context_dim": 64,
  "remix_basis_size": 64,
  "remixed_linear_kwargs": {
    "use_basis_gate": true,
    "use_output_gate": true,
    "use_context": true
  },
  "num_experts": 8,
  "total_embed_dim": 64,
  "router_dim": 64,
  "capacity_factor": 1.0,
  "use_sparse_top_k": false,
  "top_k": 1,
  "routing_mode": "token_choice",
  "context_window": -1,
  "causal": true,
  "use_expert_mlp": true,
  "use_output_projection": true,
  "use_expert_bias": false,
  "dropout": 0.0,
  "use_shared_base": false,
  "shared_base_dim": 128,
  "use_vocab_pri

/home/seqaeon/Downloads/nanochat/nanochat/tokenizer.py:405: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_bytes = torch.load(f, map_location=device)


...

2026-03-08 23:04:30,178 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/model_000020.pt
2026-03-08 23:04:30,180 - nanochat.checkpoint_manager - INFO - Saved metadata to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/meta_000020.json


<|bos|>My favorite color is,,,,,,,,,,,,,,,,
<|bos|>If 5*x + 3 = 13, then x is,,,,,,,,,,,,,,,,


2026-03-08 23:04:30,282 - nanochat.checkpoint_manager - INFO - Saved optimizer state to: /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20/optim_000020_rank0.pt


Peak memory usage: 610.52MiB
Total training time: 0.09m
Minimum validation bpb: 2.544282


step,▁▁█
total_training_flops,▁▁█
total_training_time,▁▁█
train/dt,▁
train/loss,▁
train/lrm,▁
train/mfu,▁
train/tok_per_sec,▁
val/bpb,█▁
step,20
total_training_flops,4355637903360


In [ ]:
# Auto-research sweep runner with 20-iteration checkpoints per configuration.
# Notes:
# - Every experiment is trained for exactly 20 iterations (for apples-to-apples comparison).
# - The sweep can be as large as needed; use `max_experiments` to cap for quick tests.
# - Sequence length and target dim are fixed at 64 for all experiments.
# - To avoid notebook memory buildup, each experiment can run in a fresh subprocess.

import copy
import csv
import gc
import glob
import itertools
import json
import os
import runpy
import subprocess
import sys
import time

from nanochat.common import get_base_dir
from nanochat.gpt import GPT, GPTConfig


def _strip_research_runtime_keys(research_kwargs: dict) -> dict:
    """Keep only model-relevant keys for GPT kwargs and silence research_* warning noise."""
    return {k: v for k, v in research_kwargs.items() if not k.startswith('research_')}


def _write_rows_tsv(rows: list[dict], out_tsv: str):
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with open(out_tsv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='	')
        writer.writeheader()
        writer.writerows(rows)


def _run_training_argv(
    argv: list[str],
    launch_mode: str = 'subprocess',
    nproc_per_node: int = 1,
    cuda_visible_devices: str | None = None,
):
    """Run one training job.

    launch_mode:
      - 'runpy': in-process execution (highest memory accumulation risk)
      - 'subprocess': fresh python process per run
      - 'torchrun': distributed launch; use nproc_per_node > 1 for multi-GPU
    """
    env = os.environ.copy()
    if cuda_visible_devices is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(cuda_visible_devices)

    if launch_mode == 'runpy':
        old_argv = sys.argv[:]
        sys.argv = argv
        try:
            runpy.run_module('scripts.base_train', run_name='__main__')
        finally:
            sys.argv = old_argv
        return

    cli_args = argv[1:]  # drop 'scripts.base_train'
    if launch_mode == 'torchrun' or nproc_per_node > 1:
        cmd = ['torchrun', f'--nproc_per_node={int(nproc_per_node)}', '-m', 'scripts.base_train', *cli_args]
    else:
        cmd = [sys.executable, '-m', 'scripts.base_train', *cli_args]

    subprocess.run(cmd, check=True, env=env)


def _cleanup_runtime_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def estimate_model_params(train_cfg: dict, research_kwargs: dict, vocab_size: int = 32768) -> int:
    depth = int(train_cfg['depth'])
    head_dim = int(train_cfg['head_dim'])
    if research_kwargs.get('use_moe', False):
        model_dim = int(research_kwargs.get('target_dim', 64))
    else:
        base_dim = depth * int(train_cfg['aspect_ratio'])
        model_dim = ((base_dim + head_dim - 1) // head_dim) * head_dim
    n_head = model_dim // head_dim

    cfg = GPTConfig(
        sequence_len=int(train_cfg['max_seq_len']),
        vocab_size=vocab_size,
        n_layer=depth,
        n_head=n_head,
        n_kv_head=n_head,
        n_embd=model_dim,
        window_pattern=train_cfg.get('window_pattern', 'SSSL'),
        **research_to_gpt_kwargs(_strip_research_runtime_keys(research_kwargs)),
    )
    with __import__('torch').device('meta'):
        model = GPT(cfg)
    return sum(p.numel() for p in model.parameters())


def latest_meta_for_tag(model_tag: str):
    ckpt_dir = os.path.join(get_base_dir(), 'base_checkpoints', model_tag)
    metas = sorted(glob.glob(os.path.join(ckpt_dir, 'meta_*.json')))
    if not metas:
        return None
    with open(metas[-1], 'r', encoding='utf-8') as f:
        return json.load(f)


def _lr_variants():
    """Yield both shared-LR and separate-LR research override packs."""
    shared_values = [0.004, 0.008, 0.012, 0.02]
    for v in shared_values:
        yield 'shared', {
            'research_embedding_lr': v,
            'research_unembedding_lr': v,
            'research_matrix_lr': v,
            'research_scaler_lr': v,
        }

    separate_values = [
        (0.02, 0.004, 0.015, 0.08),
        (0.03, 0.006, 0.02, 0.12),
        (0.015, 0.003, 0.01, 0.06),
        (0.01, 0.002, 0.008, 0.04),
    ]
    for e_lr, u_lr, m_lr, s_lr in separate_values:
        yield 'separate', {
            'research_embedding_lr': e_lr,
            'research_unembedding_lr': u_lr,
            'research_matrix_lr': m_lr,
            'research_scaler_lr': s_lr,
        }


def make_experiment_grid(max_experiments: int | None = None) -> list[dict]:
    phase_defs = [
        {'name': 'phase1_moe_only', 'use_moe': True,  'use_perm': False, 'use_remixed_linear': False},
        {'name': 'phase2_moe_perm', 'use_moe': True,  'use_perm': True,  'use_remixed_linear': False},
        {'name': 'phase3_remixed',  'use_moe': False, 'use_perm': False, 'use_remixed_linear': True},
    ]

    base_axes = list(itertools.product(
        [1.0, 2.0, 3.0, 5.0, 8.0, 10.0],        # MOE scale
        [8, 16, 24, 32],                        # num_experts
        [0.0, 0.05, 0.1],                       # research_warmup
        [False, True],                          # research_one_cycle
        [0.8, 0.9],                             # research_adam_beta1
        [0.95, 0.99],                           # research_adam_beta2
    ))
    lr_axes = list(_lr_variants())

    experiments = []
    for phase in phase_defs:
        for i, (moe_scale, num_experts, warmup, one_cycle, b1, b2) in enumerate(base_axes):
            lr_mode, lr_pack = lr_axes[i % len(lr_axes)]
            exp_idx = len(experiments)
            k = exp_idx % 4
            experiments.append({
                'phase': phase['name'],
                'model_tag': f"auto_research_{exp_idx:04d}_{phase['name']}",
                'MOE_SCALE': moe_scale,
                'lr_mode': lr_mode,
                'research': {
                    'use_moe': phase['use_moe'],
                    'use_perm': phase['use_perm'],
                    'use_remixed_linear': phase['use_remixed_linear'],
                    'target_dim': 64,
                    'router_dim': 64,
                    'context_dim': 64,
                    'linear_basis_size': 64,
                    'num_experts': int(num_experts),
                    'selection_mode': 'soft',
                    'allow_replacement': True,
                    'research_warmup': float(warmup),
                    'research_one_cycle': bool(one_cycle),
                    'research_adam_beta1': float(b1),
                    'research_adam_beta2': float(b2),
                    'research_use_pos_embed': (k in [1, 3]),
                    'use_pos_embed': (k in [1, 3]),
                    'moe_use_abs_pos_embed': (k in [0, 2]),
                    **lr_pack,
                },
            })

            if max_experiments is not None and len(experiments) >= max_experiments:
                return experiments
    return experiments


AUTO_RESEARCH_OUT = 'research_sweep_results.tsv'

def run_auto_research_sweep(
    base_train_cfg: dict,
    out_tsv: str = AUTO_RESEARCH_OUT,
    max_experiments: int | None = None,
    flush_every: int = 20,
    launch_mode: str = 'subprocess',
    nproc_per_node: int = 1,
    cuda_visible_devices: str | None = None,
):
    rows = []
    experiments = make_experiment_grid(max_experiments=max_experiments)
    print(f"Planned experiments: {len(experiments)}")

    for i, exp in enumerate(experiments, start=1):
        train_cfg = copy.deepcopy(base_train_cfg)
        research_cfg = copy.deepcopy(exp['research'])

        # Fixed per your requirement for sweep comparability
        train_cfg['max_seq_len'] = 64
        train_cfg['num_iterations'] = 20
        train_cfg['model_tag'] = exp['model_tag']
        train_cfg['run'] = f"{base_train_cfg.get('run', 'nb')}-{exp['model_tag']}"

        model_params = estimate_model_params(train_cfg, research_cfg)
        argv = research_train_argv(train_cfg, research_cfg, moe_scale=exp['MOE_SCALE'])

        print(f"\n[{i}/{len(experiments)}] {exp['phase']} | tag={exp['model_tag']} | params={model_params:,} | lr_mode={exp['lr_mode']} | launch={launch_mode} nproc={nproc_per_node}")

        start = time.time()
        status = 'ok'
        error_message = ''
        try:
            _run_training_argv(
                argv,
                launch_mode=launch_mode,
                nproc_per_node=nproc_per_node,
                cuda_visible_devices=cuda_visible_devices,
            )
        except Exception as e:
            status = 'error'
            error_message = repr(e)
            print(f"Run failed: {error_message}")
        finally:
            _cleanup_runtime_memory()

        duration_s = round(time.time() - start, 2)

        meta = latest_meta_for_tag(exp['model_tag'])
        final_val_bpb = None if meta is None else meta.get('val_bpb')

        rows.append({
            'idx': i,
            'phase': exp['phase'],
            'model_tag': exp['model_tag'],
            'status': status,
            'duration_s': duration_s,
            'model_params': model_params,
            'val_bpb': final_val_bpb,
            'MOE_SCALE': exp['MOE_SCALE'],
            'lr_mode': exp['lr_mode'],
            'launch_mode': launch_mode,
            'nproc_per_node': nproc_per_node,
            'num_iterations': train_cfg['num_iterations'],
            'use_moe': research_cfg['use_moe'],
            'use_perm': research_cfg['use_perm'],
            'use_remixed_linear': research_cfg['use_remixed_linear'],
            'max_seq_len': train_cfg['max_seq_len'],
            'target_dim': research_cfg['target_dim'],
            'num_experts': research_cfg['num_experts'],
            'research_warmup': research_cfg['research_warmup'],
            'research_one_cycle': research_cfg['research_one_cycle'],
            'research_embedding_lr': research_cfg['research_embedding_lr'],
            'research_unembedding_lr': research_cfg['research_unembedding_lr'],
            'research_matrix_lr': research_cfg['research_matrix_lr'],
            'research_scaler_lr': research_cfg['research_scaler_lr'],
            'research_adam_beta1': research_cfg['research_adam_beta1'],
            'research_adam_beta2': research_cfg['research_adam_beta2'],
            'moe_use_abs_pos_embed': research_cfg['moe_use_abs_pos_embed'],
            'research_use_pos_embed': research_cfg['research_use_pos_embed'],
            'use_pos_embed': research_cfg.get('use_pos_embed', False),
            'error': error_message,
            'argv': ' '.join(argv),
        })

        if flush_every > 0 and (i % flush_every == 0):
            _write_rows_tsv(rows, out_tsv)
            print(f"Checkpoint save ({i} experiments): {out_tsv}")

    _write_rows_tsv(rows, out_tsv)
    print(f"Saved sweep log: {out_tsv}")
    return rows


# Examples:
# quick smoke run (still 20 iters each) in fresh subprocesses:
# rows = run_auto_research_sweep(TRAINING_CONFIG, max_experiments=6, flush_every=2, launch_mode='subprocess', nproc_per_node=1)
# full larger sweep (single GPU subprocess):
# rows = run_auto_research_sweep(TRAINING_CONFIG, max_experiments=None, flush_every=20, launch_mode='subprocess', nproc_per_node=1)
# full larger sweep on 2 GPUs (DDP via torchrun):
# rows = run_auto_research_sweep(TRAINING_CONFIG, max_experiments=None, flush_every=20, launch_mode='torchrun', nproc_per_node=2, cuda_visible_devices='0,1')
# rows[:2]

